# Flow Matching（条件流匹配 CFM）玩具示例

在 **2D 合成数据** 上训练速度场 `v_θ(x, t)`，从噪声分布流向数据分布。不依赖外部数据集；若以后要接图像，只需把 `x` 换成图像张量并把 MLP 换成 U-Net 等即可，损失形式类似。

**路径**：线性插值 `x_t = (1-t)x_0 + t x_1`，其中 `x_0 ~ N(0,I)`，`x_1 ~ p_data`，`t ~ U(0,1)`。  
**目标速度**（该 OT 路径上为常向量）：`u = x_1 - x_0`。  
**损失**：`E[||v_θ(x_t, t) - (x_1 - x_0)||²]`。

**采样**：从 `x_0 ~ N(0,I)` 出发，用 ODE `dx/dt = v_θ(x, t)`（这里用简单欧拉法）积分到 `t=1`。

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. 合成数据 `p_data`（8 个高斯混合，环形）

In [ ]:
def sample_data(batch_size, device=device):
    """从环形 8-GMM 采样 x1 ~ p_data，shape (B, 2)。"""
    angles = torch.linspace(0, 2 * math.pi, 9, device=device)[:-1]
    centers = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 2.0
    k = centers.shape[0]
    idx = torch.randint(0, k, (batch_size,), device=device)
    c = centers[idx]
    noise = torch.randn(batch_size, 2, device=device) * 0.15
    return c + noise


def sample_noise(batch_size, device=device):
    return torch.randn(batch_size, 2, device=device)

## 2. 速度场网络 `v_θ(x, t)`

In [ ]:
class VelocityMLP(nn.Module):
    """输入 (x, t)，输出与 x 同维的速度向量。"""

    def __init__(self, dim=2, hidden=256, n_layers=5):
        super().__init__()
        self.dim = dim
        layers = []
        in_f = dim + 1  # concat t
        for i in range(n_layers - 1):
            layers += [nn.Linear(in_f if i == 0 else hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        """
        x: (B, dim), t: (B,) 或 (B,1)
        """
        if t.dim() == 1:
            t = t.unsqueeze(-1)
        inp = torch.cat([x, t], dim=-1)
        return self.net(inp)

## 3. 条件流匹配训练步（单 batch）

In [ ]:
def cfm_loss(model, x1, x0, t):
    """
    x1: 数据点 (B,2), x0: 噪声 (B,2), t: (B,) in [0,1]
    x_t = (1-t)*x0 + t*x1, target = x1 - x0
    """
    t_expand = t.view(-1, 1)
    x_t = (1.0 - t_expand) * x0 + t_expand * x1
    target = x1 - x0
    pred = model(x_t, t)
    return torch.mean((pred - target) ** 2)

## 4. 采样：ODE `dx/dt = v(x,t)`，从 t=0 到 t=1（欧拉法）

In [ ]:
@torch.no_grad()
def sample_ode(model, n, n_steps=200, device=device):
    """x0 ~ N(0,I)，沿 ODE 积分得到近似 x1。"""
    model.eval()
    x = torch.randn(n, 2, device=device)
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t = torch.full((n,), i * dt, device=device)
        v = model(x, t)
        x = x + v * dt
    return x.cpu().numpy()

## 5. 训练循环

In [ ]:
dim = 2
batch_size = 512
n_iters = 8000
lr = 1e-3

model = VelocityMLP(dim=dim, hidden=256, n_layers=5).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=lr)

losses = []
for it in range(n_iters):
    model.train()
    x1 = sample_data(batch_size, device)
    x0 = sample_noise(batch_size, device)
    t = torch.rand(batch_size, device=device)

    loss = cfm_loss(model, x1, x0, t)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

    if (it + 1) % 1000 == 0:
        print(f"iter {it+1}/{n_iters}  loss={loss.item():.6f}")

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("MSE")
plt.title("Flow matching training loss")
plt.yscale("log")
plt.tight_layout()
plt.show()

## 6. 可视化：生成样本 vs 真实数据

In [ ]:
real = sample_data(3000, device).cpu().numpy()
gen = sample_ode(model, 3000, n_steps=200, device=device)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(real[:, 0], real[:, 1], s=3, alpha=0.35)
ax[0].set_title("p_data (synthetic)")
ax[0].set_aspect("equal")
ax[1].scatter(gen[:, 0], gen[:, 1], s=3, alpha=0.35, c="C1")
ax[1].set_title("generated (ODE from noise)")
ax[1].set_aspect("equal")
plt.tight_layout()
plt.show()

---

**接图像数据时**：把 `x` 变为 `(B,C,H,W)`，`VelocityMLP` 换成带时间 embedding 的 U-Net；`x_t` 与 `target` 仍在同一形状上做逐像素 MSE（或加权重）。采样时对图像空间做相同 ODE 步进即可。